# Predicting loan default at approval time (Lending Club)

This notebook walks through the whole project step by step. Each step calls the same code as the scripts in `src/`, so the notebook and the pipeline always agree.

**Question:** on the day a loan is approved, can we tell which borrowers will not repay?

| Step | What happens | Code |
|---|---|---|
| 1 | Extract: download 2.26M loans from Kaggle | `src/download_data.py` |
| 2 | Transform + load: select, clean, engineer features | `src/etl.py` |
| 3 | EDA with statistical tests | `src/eda.py` |
| 4 | Train 4 models, compare, test the winner on 2015 loans | `src/train.py` |

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

sys.path.insert(0, str(Path.cwd().parent / "src"))
import config, download_data, etl, eda, train  # noqa: E402

pd.set_option("display.max_columns", 50)
print("Project folder:", config.PROJECT_DIR)

Project folder: /Users/sanketpatil/ml-dl-ai-models-hub/projects/01-lending-club-default


## Step 1: Extract

Downloads the Kaggle dataset (about 1.7 GB) and keeps only the accepted-loans file. If the file is already there, this does nothing, so it's safe to re-run.

In [ ]:
download_data.main()

## Step 2: Transform and load (ETL)

We keep **36-month loans issued 2012 to 2015 that ended as Fully Paid or Charged Off**, so every loan has a final answer. We load **only the columns known on the day of approval**, which avoids data leakage. Then we fix messy values and add credit-analyst features.

Watch how many rows survive each filter in the output below.

In [ ]:
etl.main()

In [ ]:
quality = json.loads((config.REPORTS_DIR / "data_quality.json").read_text())
loans = pd.read_csv(config.CLEAN_FILE)
print(f"{len(loans):,} loans x {loans.shape[1]} columns")
display(loans.head())
pd.Series(quality["missing_share_by_column"], name="share missing").to_frame()

In [ ]:
loans.describe().T.round(2)

## Step 3: Exploratory analysis with statistical tests

* **Chi-square + Cramér's V** for categorical features vs default
* **Mann-Whitney U + rank-biserial effect size** for numeric features (skewed data, so no t-test)

With this many rows nearly every p-value is tiny. **Look at the effect sizes** to see which features really matter.

In [ ]:
eda.main()

In [ ]:
for name in ["default_rate_by_grade", "default_rate_by_year", "fico_by_outcome"]:
    display(Image(filename=str(config.FIGURES_DIR / f"{name}.png")))

## Step 4: Train and compare four models

1. **Out-of-time split:** train on 2012-2014 loans, test on 2015 loans.
2. **Tune** Logistic Regression, Decision Tree, Random Forest and Gradient Boosting with 3-fold cross-validation (ROC AUC).
3. **Compare** them on ROC AUC, PR AUC, F1, precision, recall, accuracy and Brier score.
4. **Choose** the best by CV ROC AUC, pick its F1-maximising threshold, then score the 2015 test set **once**.

This is the slowest step (a few minutes).

In [ ]:
train.main()

### Model comparison (cross-validated, training years only)

In [ ]:
pd.read_csv(config.REPORTS_DIR / "model_comparison_cv.csv")

In [ ]:
display(Image(filename=str(config.FIGURES_DIR / "roc_pr_curves.png")))

### The winner on the 2015 test set

In [ ]:
results = json.loads((config.REPORTS_DIR / "test_results.json").read_text())
print("Winner:", results["winner"], results["winner_params"])
pd.DataFrame({"threshold 0.5": results["test_metrics_threshold_0.5"],
              "tuned threshold": results["test_metrics_tuned_threshold"]})

In [ ]:
display(Image(filename=str(config.FIGURES_DIR / "confusion_matrices.png")))
display(Image(filename=str(config.FIGURES_DIR / "feature_importance.png")))

### Business view: risk deciles

Loans are sorted from riskiest to safest and cut into 10 groups. `cumulative_defaults_caught` shows how many of all defaults sit in the riskiest X% of loans. Compare `predicted_pd` with `actual_default_rate` to check whether the probabilities are calibrated.

In [ ]:
display(Image(filename=str(config.FIGURES_DIR / "risk_deciles.png")))
pd.read_csv(config.REPORTS_DIR / "test_risk_deciles.csv")

## Conclusion

_Write 3-4 sentences here after the run: which model won and by how much, what the riskiest 10-20% of loans contain, and the top features. The same summary goes into the README._